In [9]:
# SEL 1: Hapus diffusers penyebab konflik dan instal pustaka pendukung
!pip uninstall -y diffusers
!pip install -q --upgrade huggingface_hub transformers optimum[onnxruntime] onnx soundfile librosa
print("Instalasi selesai!")

Found existing installation: diffusers 0.40.0
Uninstalling diffusers-0.40.0:
  Successfully uninstalled diffusers-0.40.0
Instalasi selesai!


In [10]:
# SEL 2: Ekspor Whisper ke ONNX langsung via Python API
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq

model_id = "openai/whisper-tiny"
save_dir = "./whisper-onnx-fp32"

print("Mengunduh dan mengekspor model ke format ONNX...")
model = ORTModelForSpeechSeq2Seq.from_pretrained(model_id, export=True)
model.save_pretrained(save_dir)

print(f"\nEkspor ONNX FP32 berhasil! File tersimpan di folder: {save_dir}")

Mengunduh dan mengekspor model ke format ONNX...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.13/dist-packages/optimum/exporters/base.py:151: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self._normalized_config = self.NORMALIZED_CONFIG_CLASS(self._config)
Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 1


Ekspor ONNX FP32 berhasil! File tersimpan di folder: ./whisper-onnx-fp32


In [12]:
# SEL 3 (FIXED): Kuantisasi Dinamis tanpa parameter optimize_model
import os
from onnxruntime.quantization import quantize_dynamic, QuantType

def quantize_whisper_onnx(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    onnx_files = [f for f in os.listdir(input_dir) if f.endswith('.onnx')]

    print("Memulai kuantisasi dinamis...")
    for file_name in onnx_files:
        input_model_path = os.path.join(input_dir, file_name)
        output_model_path = os.path.join(output_dir, file_name.replace(".onnx", "_int8.onnx"))

        print(f"Memproses: {file_name}...")
        quantize_dynamic(
            model_input=input_model_path,
            model_output=output_model_path,
            weight_type=QuantType.QUInt8
        )

        # Kalkulasi perbandingan ukuran
        ori_size = os.path.getsize(input_model_path) / (1024 * 1024)
        new_size = os.path.getsize(output_model_path) / (1024 * 1024)
        print(f"-> Ukuran {file_name} turun dari {ori_size:.2f} MB ke {new_size:.2f} MB")

    print(f"\nProses selesai! Model INT8 tersimpan di direktori: {output_dir}")

# Jalankan fungsi
quantize_whisper_onnx("./whisper-onnx-fp32", "./whisper-onnx-int8")

Memulai kuantisasi dinamis...
Memproses: decoder_model.onnx...


-> Ukuran decoder_model.onnx turun dari 188.83 MB ke 47.47 MB
Memproses: decoder_with_past_model.onnx...


-> Ukuran decoder_with_past_model.onnx turun dari 184.31 MB ke 46.31 MB
Memproses: encoder_model.onnx...
-> Ukuran encoder_model.onnx turun dari 31.36 MB ke 9.62 MB

Proses selesai! Model INT8 tersimpan di direktori: ./whisper-onnx-int8


In [13]:
# SEL 4: Zip file hasil kuantisasi dan unduh
import shutil
from google.colab import files

# Mengompres folder menjadi file zip agar mudah diunduh
shutil.make_archive('whisper-int8-model', 'zip', './whisper-onnx-int8')

print("Mengunduh model ke komputer lokal...")
files.download('whisper-int8-model.zip')

Mengunduh model ke komputer lokal...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>